# Attack lab: breaking a classifier with gradients and without

The optimisation tools of this class have two faces. **Gradient descent** trains a model by moving the *weights* against the
gradient of the loss. The same gradient with respect to the *input* tells an attacker how to change one example so that the model
is wrong. When there is no gradient (a black-box API) we fall back to **derivative-free optimisation**, e.g. particle swarms.

| | white-box | black-box |
|:--|:--|:--|
| knowledge | architecture and weights | only the output (probabilities) for a chosen input |
| tool | gradient of the loss w.r.t. the input | search: PSO, genetic algorithms, random search |
| cost | 1 backward pass per step | thousands of *queries* |
| example here | **FGSM** and **PGD** | **PSO** |

We attack a small **image** classifier (8x8 handwritten digits) because perturbations are easy to see and bound. In the spam
notebooks (`01-spam/notebook_07`) the same ideas are applied to *text*, where the perturbation must stay a valid, working message.

> Lab use only: we attack models that we trained ourselves.

In [ ]:
import os

os.environ.setdefault("KERAS_BACKEND", "jax")

import jax
import jax.numpy as jnp
import keras
import numpy as np
import pyBlindOpt.pso as pso
from matplotlib import pyplot as plt
from sklearn.datasets import load_digits
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

plt.rcParams["figure.dpi"] = 100
keras.utils.set_random_seed(42)

X, y = load_digits(return_X_y=True)
X = (X / 16.0).astype("float32")  # pixels in [0, 1]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)
print(X_train.shape, X_test.shape)

## 1. Two victims

* **Logistic regression** (multinomial): linear, gradient in closed form.
* **A small Keras MLP** (Keras 3 with the JAX backend, so we can differentiate through it with `jax.grad`).

In [ ]:
lr = LogisticRegression(max_iter=3000).fit(X_train, y_train)

mlp = keras.Sequential(
    [
        keras.layers.Input(shape=(64,)),
        keras.layers.Dense(64, activation="relu"),
        keras.layers.Dense(64, activation="relu"),
        keras.layers.Dense(10),  # logits
    ]
)
mlp.compile(
    optimizer=keras.optimizers.Adam(1e-2),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"],
)
mlp.fit(X_train, y_train, epochs=40, batch_size=64, verbose=0)
print(f"clean accuracy   LR: {lr.score(X_test, y_test):.3f}   MLP: {mlp.evaluate(X_test, y_test, verbose=0)[1]:.3f}")

## 2. White-box: FGSM (Fast Gradient Sign Method)

One step in the direction that **increases the loss** the most under an $\ell_\infty$ budget $\varepsilon$ (every pixel may move by at
most $\varepsilon$):

$$
\mathbf{x}_{adv}=\operatorname{clip}_{[0,1]}\!\Big(\mathbf{x}+\varepsilon\cdot\operatorname{sign}\big(\nabla_{\mathbf{x}}J(\theta,\mathbf{x},y)\big)\Big)
$$

For logistic regression, $\nabla_{\mathbf{x}}J=W^\top(\mathbf{p}-\mathbf{e}_y)$ (see the calculus notebook). For the MLP we let JAX do it.

In [ ]:
def grad_lr(xb, yb):
    p = lr.predict_proba(xb)
    p[np.arange(len(yb)), yb] -= 1.0
    return p @ lr.coef_  # (n, 10) @ (10, 64)


tv = [v.value for v in mlp.trainable_variables]
ntv = [v.value for v in mlp.non_trainable_variables]


def _loss(xb, yb):
    logits, _ = mlp.stateless_call(tv, ntv, xb, training=False)
    return keras.losses.sparse_categorical_crossentropy(yb, logits, from_logits=True).sum()


_grad_mlp = jax.jit(jax.grad(_loss))
_logits_mlp = jax.jit(lambda xb: mlp.stateless_call(tv, ntv, xb, training=False)[0])


def grad_mlp(xb, yb):
    return np.asarray(_grad_mlp(jnp.asarray(xb), jnp.asarray(yb)))


def predict_lr(xb):
    return lr.predict(xb)


def predict_mlp(xb):
    return np.asarray(_logits_mlp(jnp.asarray(xb)).argmax(axis=1))


def fgsm(grad, xb, yb, eps):
    return np.clip(xb + eps * np.sign(grad(xb, yb)), 0.0, 1.0)


eps_grid = [0.0, 0.05, 0.1, 0.15, 0.2, 0.3, 0.4]
print(f"{'eps':>6}{'LR acc':>9}{'MLP acc':>9}   ({'random sign noise, same eps'})")
rng = np.random.default_rng(0)
rows = []
for eps in eps_grid:
    a_lr = (predict_lr(fgsm(grad_lr, X_test, y_test, eps)) == y_test).mean()
    a_mlp = (predict_mlp(fgsm(grad_mlp, X_test, y_test, eps)) == y_test).mean()
    noisy = np.clip(X_test + eps * rng.choice([-1, 1], X_test.shape), 0, 1)
    a_rnd = (predict_mlp(noisy) == y_test).mean()
    rows.append((eps, a_lr, a_mlp, a_rnd))
    print(f"{eps:6.2f}{a_lr:9.3f}{a_mlp:9.3f}   {a_rnd:.3f}")

fig, ax = plt.subplots(figsize=(6, 3.6))
ax.plot(*zip(*[(r[0], r[1]) for r in rows]), "o-", label="LR under FGSM")
ax.plot(*zip(*[(r[0], r[2]) for r in rows]), "o-", label="MLP under FGSM")
ax.plot(*zip(*[(r[0], r[3]) for r in rows]), "k:", label="MLP under random noise")
ax.set(xlabel=r"$\ell_\infty$ budget $\varepsilon$", ylabel="accuracy")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

**Random noise of the same size barely matters; a *gradient-guided* perturbation of the same size destroys the accuracy.** The
direction is what counts, not the amount of noise.

In [ ]:
idx = [0, 1, 2, 3, 4]
xb, yb = X_test[idx], y_test[idx]
adv = fgsm(grad_mlp, xb, yb, 0.25)
fig, ax = plt.subplots(3, len(idx), figsize=(8, 5))
for j in range(len(idx)):
    ax[0, j].imshow(xb[j].reshape(8, 8), cmap="gray_r", vmin=0, vmax=1)
    ax[0, j].set_title(f"clean: {predict_mlp(xb[j : j + 1])[0]}")
    ax[1, j].imshow((adv[j] - xb[j]).reshape(8, 8), cmap="RdBu", vmin=-0.3, vmax=0.3)
    ax[1, j].set_title("perturbation")
    ax[2, j].imshow(adv[j].reshape(8, 8), cmap="gray_r", vmin=0, vmax=1)
    ax[2, j].set_title(f"FGSM: {predict_mlp(adv[j : j + 1])[0]}")
for a in ax.ravel():
    a.axis("off")
plt.tight_layout()
plt.show()

## 3. Stronger white-box: PGD (iterated FGSM)

FGSM takes one big step. **Projected Gradient Descent** takes several small ones, projecting back into the $\varepsilon$-ball
and the valid pixel range after each. It is the standard *strong* first-order attack, and the reference when testing defences.

In [ ]:
def pgd(grad, xb, yb, eps, steps=20, alpha=None):
    alpha = alpha or 2.5 * eps / steps
    x_adv = xb + rng.uniform(-eps, eps, xb.shape).astype("float32")
    for _ in range(steps):
        x_adv = x_adv + alpha * np.sign(grad(np.clip(x_adv, 0, 1), yb))
        x_adv = np.clip(np.clip(x_adv, xb - eps, xb + eps), 0.0, 1.0)
    return x_adv


print(f"{'eps':>6}{'MLP FGSM':>10}{'MLP PGD':>10}")
for eps in (0.05, 0.1, 0.2):
    a_f = (predict_mlp(fgsm(grad_mlp, X_test, y_test, eps)) == y_test).mean()
    a_p = (predict_mlp(pgd(grad_mlp, X_test, y_test, eps)) == y_test).mean()
    print(f"{eps:6.2f}{a_f:10.3f}{a_p:10.3f}")

## 4. Transfer: crafting on a surrogate, hitting a different model

What if the attacker cannot differentiate the victim? Build a **surrogate** (here: the logistic regression), craft the
adversarial examples on it, and *hope they transfer*. They often do, because different models trained on the same task learn similar
decision boundaries.

In [ ]:
print(f"{'eps':>6}{'crafted on LR -> LR':>21}{'crafted on LR -> MLP':>22}")
for eps in (0.1, 0.2, 0.3):
    adv_lr = fgsm(grad_lr, X_test, y_test, eps)
    print(f"{eps:6.2f}{(predict_lr(adv_lr) == y_test).mean():21.3f}{(predict_mlp(adv_lr) == y_test).mean():22.3f}")

## 5. Black-box: particle swarm optimisation

Now the attacker only sees the **probability of the true class** returned by a scoring API. We search for a perturbation
$\boldsymbol\delta\in[-\varepsilon,\varepsilon]^{64}$ that **minimises** that probability with *particle swarm optimisation*
(the derivative-free optimiser of `notebook_03`). Cost: one *query* per particle per iteration.

In [ ]:
_probs_mlp = jax.jit(lambda xb: jax.nn.softmax(_logits_mlp(xb), axis=1))


def pso_attack(x, label, eps, n_pop=30, n_iter=40, seed=0):
    x = x.astype("float64")
    queries = 0

    def objective(delta):
        nonlocal queries
        queries += 1
        xa = np.clip(x + delta, 0, 1).astype("float32")
        return float(_probs_mlp(jnp.asarray(xa[None]))[0, label])

    bounds = np.array([(-eps, eps)] * x.size)
    rng_ = np.random.default_rng(seed)
    pop = rng_.uniform(-eps, eps, (n_pop, x.size))
    best, score, _ = pso.particle_swarm_optimization(
        objective, bounds, population=pop, n_iter=n_iter, verbose=False, debug=True
    )
    return np.clip(x + best, 0, 1).astype("float32"), queries


n_attack = 30
eps = 0.2
xb, yb = X_test[:n_attack], y_test[:n_attack]
correct = predict_mlp(xb) == yb
results = [pso_attack(xb[i], yb[i], eps, seed=i) for i in range(n_attack)]
adv_pso = np.array([r[0] for r in results])
q = int(np.mean([r[1] for r in results]))
adv_fgsm = fgsm(grad_mlp, xb, yb, eps)
adv_rand = np.clip(xb + eps * rng.choice([-1, 1], xb.shape), 0, 1).astype("float32")

print(f"attacking {correct.sum()} correctly classified digits, eps={eps}")
print(f"{'attack':22}{'success rate':>14}{'cost':>22}")
for name, adv, cost in (
    ("random noise", adv_rand, "0 queries"),
    ("FGSM (white-box)", adv_fgsm, "1 gradient"),
    ("PSO (black-box)", adv_pso, f"{q} queries / image"),
):
    print(f"{name:22}{(predict_mlp(adv)[correct] != yb[correct]).mean():14.2f}{cost:>22}")

## What to take away

* The **gradient of the loss with respect to the input** is the attacker's most valuable quantity; it is the *same object* as the
  training gradient, evaluated on a different variable.
* A tiny, *structured* perturbation is enough; random noise of the same size is harmless. **Robustness to noise is not robustness to an adversary.**
* Without gradients the attacker pays in **queries** (~1,200 here), but derivative-free search still works. Limiting and monitoring
  queries is a *defence*, not a guarantee.
* Perturbations *transfer* between models, so hiding the weights is not enough.
* For **discrete** inputs (text, binaries) the perturbation must stay valid: the attack becomes a *search over edits* (see the spam
  notebook 07).

## Exercises

1. Plot the adversarial accuracy of the MLP for $\varepsilon$ up to 0.6. At which $\varepsilon$ does the *image* stop looking like a digit to you?
2. Use the $\ell_2$ norm instead of $\ell_\infty$: $\mathbf{x}_{adv}=\mathbf{x}+\varepsilon\,\nabla/\lVert\nabla\rVert_2$. Which budget is easier to defend?
3. **Defence.** Adversarial training: retrain the MLP on a mix of clean and FGSM examples (generated on the fly). Re-measure the PGD accuracy. What happens to the clean accuracy?
4. Increase the PSO population and iterations. How does the success rate scale with the number of queries?